In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

print("=" * 55)
print("DESCRIBE HISTORY + OPTIMIZE + VACUUM NOTE")
print("=" * 55)

DESCRIBE HISTORY + OPTIMIZE + VACUUM NOTE


In [0]:
# DESCRIBE HISTORY on all key tables
tables_to_inspect = [
    "bronze_insclm.bronze_claims",
    "bronze_insclm.bronze_claim_status_updates",
    "bronze_insclm.bronze_policy_master",
    "bronze_insclm.bronze_customer_master",
    "silver_insclm.silver_claims_fact",
    "silver_insclm.silver_customer_dim",
    "silver_insclm.silver_policy_dim",
    "silver_insclm.silver_claim_status_history",
    "gold_insclm.gold_claim_summary",
    "gold_insclm.gold_policy_history_summary",
    "gold_insclm.gold_suspicious_claim_summary",
    "rejected_insclm.rejected_claims",
    "rejected_insclm.rejected_status_updates",
    "rejected_insclm.rejected_policy",
    "rejected_insclm.rejected_customers",
]

print("\n📋 DESCRIBE HISTORY — all tables\n")

for table in tables_to_inspect:
    try:
        print(f"\n{'='*55}")
        print(f"TABLE: {table}")
        print(f"{'='*55}")
        DeltaTable.forName(spark, table) \
            .history() \
            .select(
                "version",
                "timestamp",
                "operation",
                "operationMetrics"
            ).show(3, truncate=False)
    except Exception as e:
        print(f"⚠️  Could not read {table}: {e}")


📋 DESCRIBE HISTORY — all tables


TABLE: bronze_insclm.bronze_claims
+-------+-------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationMetrics                                                                                                                               |
+-------+-------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-05-21 14:53:04|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 31340, numDeletionVectorsRemoved -> 0, numOutputRows -> 2200, numOutputBytes -> 31340}|
|3      |2026-05-21 14:52:09|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 

In [0]:
# OPTIMIZE on key tables
print("\n" + "=" * 55)
print("OPTIMIZE — compacting small files")
print("=" * 55)
print("Merges small Parquet files into fewer")
print("larger files for faster reads.\n")

optimize_tables = [
    "silver_insclm.silver_claims_fact",
    "silver_insclm.silver_policy_dim",
    "silver_insclm.silver_claim_status_history",
    "gold_insclm.gold_claim_summary",
    "gold_insclm.gold_suspicious_claim_summary",
]

for table in optimize_tables:
    try:
        start = datetime.now()
        print(f"📥 Optimizing {table}...")
        spark.sql(f"OPTIMIZE {table}")
        end = datetime.now()
        print(f"✅ Done in {(end-start).seconds}s")
    except Exception as e:
        print(f"⚠️  Could not optimize {table}: {e}")


OPTIMIZE — compacting small files
Merges small Parquet files into fewer
larger files for faster reads.

📥 Optimizing silver_insclm.silver_claims_fact...
✅ Done in 1s
📥 Optimizing silver_insclm.silver_policy_dim...
✅ Done in 0s
📥 Optimizing silver_insclm.silver_claim_status_history...
✅ Done in 0s
📥 Optimizing gold_insclm.gold_claim_summary...
✅ Done in 0s
📥 Optimizing gold_insclm.gold_suspicious_claim_summary...
✅ Done in 0s


In [0]:
# Verify table details after OPTIMIZE
print("\n📋 Table details after OPTIMIZE:\n")

detail_tables = [
    "silver_insclm.silver_claims_fact",
    "silver_insclm.silver_policy_dim",
    "gold_insclm.gold_claim_summary",
]

for table in detail_tables:
    try:
        print(f"\n{table}:")
        spark.sql(f"DESCRIBE DETAIL {table}") \
            .select(
                "numFiles",
                "sizeInBytes",
                "location"
            ).show(truncate=False)
    except Exception as e:
        print(f"⚠️  {table}: {e}")


📋 Table details after OPTIMIZE:


silver_insclm.silver_claims_fact:
+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|numFiles|sizeInBytes|location                                                                                                                                                                                           |
+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1       |64233      |abfss://unity-catalog-storage@dbstoragewtm4yosnnwbtm.dfs.core.windows.net/7405614049100195/__unitystorage/catalogs/e6ace269-b861-419b-b411-dc9adc645854/tables/8655491c-89f5-4951-bb26-d9f90fad3a81|
+--------+-----------+---------------------------------

In [0]:
# Set retention properties on key tables
print("\n📥 Setting retention properties...")
print("   Minimum 730 days for insurance compliance\n")

retention_tables = [
    "silver_insclm.silver_policy_dim",
    "silver_insclm.silver_claims_fact",
    "silver_insclm.silver_claim_status_history",
    "gold_insclm.gold_suspicious_claim_summary",
]

for table in retention_tables:
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            SET TBLPROPERTIES (
                'delta.deletedFileRetentionDuration'
                    = 'interval 730 days',
                'delta.logRetentionDuration'
                    = 'interval 730 days'
            )
        """)
        print(f"✅ {table} → retention set to 730 days")
    except Exception as e:
        print(f"⚠️  {table}: {e}")


📥 Setting retention properties...
   Minimum 730 days for insurance compliance

✅ silver_insclm.silver_policy_dim → retention set to 730 days
✅ silver_insclm.silver_claims_fact → retention set to 730 days
✅ silver_insclm.silver_claim_status_history → retention set to 730 days
✅ gold_insclm.gold_suspicious_claim_summary → retention set to 730 days


In [0]:
# VACUUM explanation — why we do NOT run it
print("\n" + "=" * 55)
print("VACUUM — WHY WE DO NOT RUN IT")
print("=" * 55)

vacuum_explanation = """
WHAT IS VACUUM?
    Permanently deletes old Parquet files older
    than the retention period (default 7 days).
    Once deleted — CANNOT be recovered.

WHY VACUUM IS DANGEROUS IN INSURANCE:

    1. REGULATORY COMPLIANCE
       IRDAI requires insurance records preserved
       for minimum 5-10 years. VACUUM destroys
       old Delta versions — compliance impossible.

    2. TIME TRAVEL IS LOST
       After VACUUM, AS OF queries fail because
       old files no longer exist.

    3. SCD TYPE 2 AUDIT TRAIL DESTROYED
       policy_dim MERGE history proves WHEN
       policies changed. VACUUM removes evidence.

    4. FRAUD INVESTIGATION
       Investigators need original claim values
       from filing date. VACUUM destroys this.

    5. NO RECOVERY
       Unlike database DELETE with rollback,
       VACUUM is permanent. No undo.

SAFE APPROACH FOR INSURANCE:
    Retention set to 730 days minimum.
    Use ADLS lifecycle policies for cost savings
    instead of VACUUM.

    If VACUUM absolutely required:
    VACUUM silver_insclm.silver_policy_dim
        RETAIN 17520 HOURS DRY RUN;
    — Always DRY RUN first to preview deletions.

BOTTOM LINE:
    VACUUM is NOT run on this insurance dataset.
    All Delta history preserved for compliance.
"""

print(vacuum_explanation)


VACUUM — WHY WE DO NOT RUN IT

WHAT IS VACUUM?
    Permanently deletes old Parquet files older
    than the retention period (default 7 days).
    Once deleted — CANNOT be recovered.

WHY VACUUM IS DANGEROUS IN INSURANCE:

    1. REGULATORY COMPLIANCE
       IRDAI requires insurance records preserved
       for minimum 5-10 years. VACUUM destroys
       old Delta versions — compliance impossible.

    2. TIME TRAVEL IS LOST
       After VACUUM, AS OF queries fail because
       old files no longer exist.

    3. SCD TYPE 2 AUDIT TRAIL DESTROYED
       policy_dim MERGE history proves WHEN
       policies changed. VACUUM removes evidence.

    4. FRAUD INVESTIGATION
       Investigators need original claim values
       from filing date. VACUUM destroys this.

    5. NO RECOVERY
       Unlike database DELETE with rollback,
       VACUUM is permanent. No undo.

SAFE APPROACH FOR INSURANCE:
    Retention set to 730 days minimum.
    Use ADLS lifecycle policies for cost savings
    instead

In [0]:
print("\n" + "=" * 55)
print("SUMMARY")
print("=" * 55)
print("✅ DESCRIBE HISTORY — 15 tables inspected")
print("✅ OPTIMIZE — 5 tables compacted")
print("✅ DESCRIBE DETAIL — file counts verified")
print("✅ Retention — 730 days set on 4 tables")
print("✅ VACUUM — documented, NOT executed")
print("=" * 55)
print("✅ NB_08 complete.")
print("   Next: Run NB_09_load_to_azure_sql")


SUMMARY
✅ DESCRIBE HISTORY — 15 tables inspected
✅ OPTIMIZE — 5 tables compacted
✅ DESCRIBE DETAIL — file counts verified
✅ Retention — 730 days set on 4 tables
✅ VACUUM — documented, NOT executed
✅ NB_08 complete.
   Next: Run NB_09_load_to_azure_sql


In [0]:
# Test SQL connection
df_test = spark.read \
    .format("jdbc") \
    .option("url", SQL_JDBC_URL) \
    .option("query", "SELECT 1 AS test") \
    .option("user", SQL_USER) \
    .option("password", SQL_PASSWORD) \
    .option("driver",
        "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

df_test.show()
print("✅ SQL connection working")

+----+
|test|
+----+
|   1|
+----+

✅ SQL connection working
